## TREINAMENTO - REDE NEURAL

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'torch'

import keras
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.metrics import (classification_report, roc_auc_score,
                             ConfusionMatrixDisplay, RocCurveDisplay)
from sklearn.inspection import permutation_importance
from sklearn.base import BaseEstimator, ClassifierMixin

from google.colab import drive
drive.mount('/content/drive')

### CARREGAR DADOS PROCESSADOS

In [ ]:
X_train = pd.read_csv('/content/drive/MyDrive/X_train_proc.csv')
X_test  = pd.read_csv('/content/drive/MyDrive/X_test_proc.csv')

y_train = pd.read_csv('/content/drive/MyDrive/y_train.csv').squeeze()
y_test  = pd.read_csv('/content/drive/MyDrive/y_test.csv').squeeze()

# Para a rodada 2: treino + teste1 juntos
X_train2 = pd.concat([X_train, X_test], ignore_index=True)
y_train2 = pd.concat([y_train, y_test], ignore_index=True)

# teste2 - carregar separado
X_test2 = pd.read_csv('/content/drive/MyDrive/X_test2_proc.csv')
y_test2 = pd.read_csv('/content/drive/MyDrive/y_test2.csv').squeeze()

print(f'X_train : {X_train.shape}')
print(f'X_test1 : {X_test.shape}')
print(f'X_train2: {X_train2.shape}  <- treino + teste1')
print(f'X_test2 : {X_test2.shape}')
print(f'\nDesbalanceamento — Taxa de abandono: {y_train.mean()*100:.1f}%')

# peso para classe minoritária
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
class_weight = {0: 1.0, 1: n_neg / n_pos}
print(f'Class weights: {class_weight}')


# MÉTRICAS DE AVALIAÇÃO

### **ACURÁCIA:** NÃO será usada como métrica principal pois o dataset é
### desbalanceado (~19% de abandono). Um modelo que prevê sempre "não abandona"
### já teria ~81% de acurácia sem aprender nada útil.

# Métricas escolhidas:

### **RECALL (Sensibilidade):** métrica mais importante para este problema.
### Queremos minimizar falsos negativos - pacientes que vão abandonar mas
### o modelo não detecta. No contexto clínico, deixar de identificar um
### paciente em risco é mais grave do que um falso alarme.

### **PRECISÃO:** proporção de pacientes sinalizados que realmente abandonam.
### Importante para não sobrecarregar a equipe de saúde com falsos alertas.

### **F1-SCORE:** média harmônica entre precisão e recall. Métrica principal
### pois equilibra os dois trade-offs acima no contexto desbalanceado.

### **ROC-AUC:** mede a capacidade discriminativa geral do modelo independente
### do threshold. Útil para comparar os dois modelos entre si.


### FUNÇÃO DE AVALIAÇÃO

In [ ]:
def avaliar(nome, y_true, y_pred, y_prob):
    print(f'\n{"="*50}')
    print(f' {nome}')
    print(f'{"="*50}')
    print(classification_report(y_true, y_pred))
    print(f'ROC-AUC: {roc_auc_score(y_true, y_prob):.4f}')

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    ConfusionMatrixDisplay.from_predictions(y_true, y_pred, ax=axes[0])
    axes[0].set_title(f'Matriz de Confusão - {nome}')
    RocCurveDisplay.from_predictions(y_true, y_prob, ax=axes[1])
    axes[1].set_title(f'Curva ROC - {nome}')
    plt.tight_layout()
    plt.show()

### DEFINIÇÃO DO MODELO

In [ ]:
def criar_modelo(n_features):
    modelo = keras.Sequential([
        keras.Input(shape=(n_features,)),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dense(1,  activation='sigmoid'),
    ])
    modelo.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return modelo

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

### RODADA 1: treino -> teste1

In [ ]:
modelo_RN = criar_modelo(X_train.shape[1])
modelo_RN.summary()

history = modelo_RN.fit(
    X_train, y_train,
    epochs=100,
    batch_size=256,
    validation_split=0.1,
    class_weight=class_weight,
    callbacks=[early_stop],
    verbose=1
)

pd.DataFrame(history.history)[['accuracy', 'val_accuracy']].plot(figsize=(8, 5))
plt.title('Curva de Aprendizado — Rede Neural (Rodada 1)')
plt.grid(True)
plt.show()

prob_RN_test1 = modelo_RN.predict(X_test).flatten()
pred_RN_test1 = (prob_RN_test1 >= 0.5).astype(int)
avaliar('Rede Neural - Teste 1', y_test, pred_RN_test1, prob_RN_test1)

### RODADA 2: treino + teste1 -> teste2

In [ ]:
modelo_RN2 = criar_modelo(X_train2.shape[1])

history2 = modelo_RN2.fit(
    X_train2, y_train2,
    epochs=100,
    batch_size=256,
    validation_split=0.1,
    class_weight=class_weight,
    callbacks=[early_stop],
    verbose=1
)

pd.DataFrame(history2.history)[['accuracy', 'val_accuracy']].plot(figsize=(8, 5))
plt.title('Curva de Aprendizado - Rede Neural (Rodada 2)')
plt.grid(True)
plt.show()

prob_RN_test2 = modelo_RN2.predict(X_test2).flatten()
pred_RN_test2 = (prob_RN_test2 >= 0.5).astype(int)
avaliar('Rede Neural - Teste 2', y_test2, pred_RN_test2, prob_RN_test2)

### EXPLICABILIDADE - PERMUTATION IMPORTANCE

In [ ]:
from sklearn.utils.estimator_checks import parametrize_with_checks
from sklearn.utils.multiclass import check_classification_targets

class KerasWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, model):
        self.model = model
        self.classes_ = np.array([0, 1])
        self._estimator_type = "classifier"
    def fit(self, X, y): return self
    def predict(self, X): return (self.model.predict(X).flatten() >= 0.5).astype(int)
    def predict_proba(self, X):
        p = self.model.predict(X).flatten()
        return np.column_stack([1-p, p])

print('\nCalculando Permutation Importance...')
result = permutation_importance(
    KerasWrapper(modelo_RN2), X_test2, y_test2,
    n_repeats=10,
    random_state=42,
    scoring='accuracy'
)

importancias = pd.Series(result.importances_mean, index=X_train.columns).sort_values()

plt.figure(figsize=(8, 7))
importancias.plot(kind='barh', color=importancias.map(lambda v: '#e74c3c' if v > 0 else '#95a5a6'))
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Permutation Importance — Rede Neural', fontsize=12)
plt.xlabel('Redução no ROC-AUC')
plt.tight_layout()
plt.show()

top_features = importancias[importancias > 0].sort_values(ascending=False).head(10).index.tolist()
print(f'\nTop features relevantes: {top_features}')

In [ ]:
teste2 = pd.read_csv('/content/drive/MyDrive/teste2.csv', low_memory=False)
print(teste2.shape)
print(teste2['ltfu'].value_counts())

### SALVAR MODELOS

In [ ]:
modelo_RN2.save('/content/drive/MyDrive/modelo_RN.keras')
joblib.dump(top_features, '/content/drive/MyDrive/top_features_RN.pkl')
print('modelo_RN.keras salvo!')